# Retail Arbitrage Finder with Scavio API

Find products you can buy on Walmart and resell on Amazon for profit. Uses the Scavio search API and LangChain to compare prices across platforms and calculate margins. A free alternative to Tactical Arbitrage and SellerAmp.

**What you will learn:**
- Search Walmart for low-priced products with ScavioWalmartSearch
- Cross-reference prices on Amazon with ScavioAmazonSearch
- Calculate profit margins after estimated fees
- Build an arbitrage opportunity report

**Prerequisites:**
- Free Scavio API key (50 free credits (one-time)): https://dashboard.scavio.dev
- OpenAI API key

**Tools used:** ScavioWalmartSearch, ScavioWalmartProduct, ScavioAmazonSearch, ScavioAmazonProduct

In [1]:
# pip install langchain langchain-openai langchain-scavio python-dotenv

In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_scavio import (
    ScavioWalmartSearch,
    ScavioWalmartProduct,
    ScavioAmazonSearch,
    ScavioAmazonProduct,
)

load_dotenv(override=True)

True

In [3]:
SYSTEM_PROMPT = """You are RetailArbitrageFinder, a product arbitrage research agent.

Workflow:
1. Take the user's product category.
2. Call ScavioWalmartSearch to find products in this category on Walmart.
3. Call ScavioWalmartProduct on the top 2 results to get Walmart prices.
4. For each Walmart product, call ScavioAmazonSearch with the exact
   product name to find it on Amazon.
5. Call ScavioAmazonProduct on the best Amazon match to get the
   Amazon selling price.
6. Calculate arbitrage opportunity and produce:

   ## Arbitrage Report: <category>

   For each product analyzed:
   ### <Product Name>
   - Walmart Price: $X.XX
   - Amazon Price: $X.XX
   - Price Difference: $X.XX
   - Estimated Amazon Fee (~15%): $X.XX
   - Estimated Profit: $X.XX
   - Margin: X%
   - Verdict: <Worth flipping / Too thin / Not found on Amazon>

   ### Summary
   - Best opportunity: <product with highest margin>
   - Total products analyzed: X
   - Profitable opportunities found: X

Rules:
- Never invent prices or product names. Only use tool output.
- Estimate Amazon referral fee at 15% of Amazon selling price.
- Call only ONE tool per step.
- Keep the final report under 400 words.
"""

In [4]:
def build_agent():
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    tools = [
        ScavioWalmartSearch(max_results=5),
        ScavioWalmartProduct(),
        ScavioAmazonSearch(max_results=3),
        ScavioAmazonProduct(),
    ]
    return create_agent(model, tools=tools, system_prompt=SYSTEM_PROMPT)

In [5]:
agent = build_agent()
result = agent.invoke({
    "messages": [{"role": "user", "content": "portable bluetooth speakers"}]
})
print(result["messages"][-1].content)

## Arbitrage Report: Portable Bluetooth Speakers

### Portable Bluetooth Speaker, Bluetooth 5.4 Wireless Speaker, IPX7 Waterproof, 30W Loud Deep Bass with 20H Playtime, 7 Color RGB Lights, TWS Pairing for Home/Party/Outdoor, Gift
- Walmart Price: $26.99
- Amazon Price: $26.99
- Price Difference: $0.00
- Estimated Amazon Fee (~15%): $4.05
- Estimated Profit: -$4.05
- Margin: -15%
- Verdict: Too thin

### Altec Lansing HydraMini 2.0 EverythingProof Bluetooth Speaker, Waterproof IP67, Floats, 12-Hour Playtime, Built-In Magnet, Carabiner & Mount, Blue White
- Walmart Price: $13.45
- Amazon Price: $19.99
- Price Difference: $6.54
- Estimated Amazon Fee (~15%): $3.00
- Estimated Profit: $3.54
- Margin: 26%
- Verdict: Worth flipping

### Summary
- Best opportunity: Altec Lansing HydraMini 2.0 EverythingProof Bluetooth Speaker with 26% margin
- Total products analyzed: 2
- Profitable opportunities found: 1

The Altec Lansing HydraMini 2.0 offers a good arbitrage opportunity with a decent margi

## Next Steps

- Scan any product category for arbitrage opportunities
- Focus on categories with high margin potential (electronics, toys, home)
- Track price fluctuations during sale events (Black Friday, Prime Day)
- Build a daily scanner that alerts you to new opportunities

**Credits used:** ~6-8 per run (Walmart search + products + Amazon search + products)